In [94]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_experimental.graph_transformers.llm import system_prompt
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader
from pinecone.models.index_description import ServerlessSpec


In [141]:
doc_1 = PyPDFLoader("./Data/Gale Encyclopedia of Medicine. Vol. 5. 2nd ed.pdf").load()
doc_2 = PyPDFLoader("./Data/Gale Encyclopedia of Medicine Vol. 3 (G-M).pdf").load()
doc_3 = PyPDFLoader("./Data/Gale Encyclopedia of Medicine Vol. 4 (N-S).pdf").load()
doc_4 = PyPDFLoader("./Data/The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf").load()
doc_5 = PyPDFLoader("./Data/The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND (1).pdf").load()

In [143]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [175]:
text_chunks_1 = text_split(doc_1)
text_chunks_2 = text_split(doc_2)
text_chunks_3 = text_split(doc_3)
text_chunks_4 = text_split(doc_4)
text_chunks_5 = text_split(doc_5)


In [176]:
import os
import dotenv
dotenv.load_dotenv()

True

In [177]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [178]:
from langchain.embeddings import HuggingFaceEmbeddings
def download_embeddings():
    embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embedding


In [179]:
embeddings = download_embeddings()


In [180]:
query_result = embeddings.embed_query("What is the purpose of medical assistant?")
print("length", len(query_result))

length 384


In [181]:
embeddings


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [190]:
from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medassistant"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        )
    )

# ✅ Get the actual index object
index = pc.Index(index_name)

In [191]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [192]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings, text_key="text")

In [193]:
vector_store.add_documents(documents=text_chunks_1)
vector_store.add_documents(documents=text_chunks_2)

['946868bb-a012-42d1-85bf-058531b58c08',
 '5695d4b5-538c-4e12-b7f4-3cfb67e6e681',
 'c128565b-b419-4c44-a703-dcc4ab21cf18',
 'ebb0f304-5e0d-4089-b77b-610c740762f0',
 '2e3a24a5-88d8-43dd-afe2-7804bf378707',
 'e17c70c5-5be0-499a-94ea-1a158b0a8ccb',
 'bd131254-857a-451e-bd44-1945e796da06',
 'e871721d-61ad-4dde-8187-74b93019071f',
 '3795c008-1d02-4373-b03d-dafc8fe1f8f1',
 '667e783d-1102-40bd-923d-bd28568880fa',
 'a1655922-6e3c-4f80-ad27-60a0e1b20e09',
 'b2813562-46f2-4029-adc9-d44cf90a37fd',
 '4dd7b667-dbc1-457f-ba9e-03051ed46b43',
 'bd7a1407-3c38-47ac-b70c-7fd414ad061f',
 'be581c51-dd03-41cd-abbe-cd0fff2e0805',
 'b97b728b-20d9-4d87-b5b8-f69087c7dcd6',
 'ebe63921-8b64-47ab-93d3-370f6b6135d6',
 '66fdc108-cf5b-495e-95f9-106ee10debc9',
 '91b57c3e-de91-4239-9249-e1b7210d7526',
 '7274e385-74fa-4356-a641-cc0de244d6eb',
 '42439506-88e0-4426-bfee-179e410286c3',
 '60e5eaa0-c1e4-42ed-b60d-6feff62903c1',
 '54d87f91-095e-4db1-9de5-1ec09043ee95',
 '1fa511fb-f09a-445c-8dcd-cee6e8fa8566',
 '8ab8e375-032a-

In [194]:
vector_store.add_documents(documents=text_chunks_3)
vector_store.add_documents(documents=text_chunks_4)

['1ddc1109-1a9c-4ef5-8eee-212cc0c5165b',
 '96347fac-78b9-4899-a6ed-6e795036678f',
 'f5020c23-2632-4269-9218-cd5a3d9f6482',
 'ead1c426-1bbb-4e01-a697-db102cd05cbc',
 '51efcaa6-7ab8-484d-a5e2-3cb85cecd621',
 '05cea249-b1a5-482c-af70-35bf408be12b',
 '3f31d382-5582-4308-a8e5-e6666981c823',
 '903399dc-cd8c-4b92-b8f6-ebf644729eca',
 'e12e82a2-a3c6-4dea-971c-f0205ba6b808',
 '9613f716-19eb-42b1-90be-e45c43336b22',
 'f6227d81-05e6-4579-b39b-1b733b5d09c1',
 '6c693931-6e64-43d8-8331-c09b5dca47a7',
 '65bc8158-7cb5-4b14-92fb-6a53a12c41a1',
 'ce4c5c9c-4aef-40c8-ad21-f1fcc65c57d1',
 'd442a53f-6756-4110-abc1-02d50344f9ff',
 '91b7ea94-0304-4cc3-8ce4-1fdf326c16cf',
 '2b3a9f8f-5e4e-4fb9-81ff-9fbeed4a65c5',
 'b7641727-7223-4e82-8d0d-2d7b166aacde',
 '15ff019f-176b-4b99-bdb7-06a8ba4090f7',
 '29b18d85-2670-432e-a409-5bfd0863355d',
 '456e2637-0238-4552-bafc-d44ff69bae34',
 'ec7a8c43-942c-45ac-af6f-c0c2b6772db6',
 '6fb2eed7-a8c8-4255-bb9c-4b51c958a72c',
 '9cdace1d-229b-42ac-a973-007397f3d752',
 '2aa8ea27-da8c-

In [195]:
vector_store.add_documents(documents=text_chunks_5)


['6c973a56-ce51-48a5-94cc-615a57b4795e',
 '5fd8ea8c-4a40-4c50-996d-a0f33a08fd48',
 '407ad3ad-d3d6-4d0a-b2e3-656c276e2cce',
 '55279cd0-c076-4e6a-a6c3-f1ef55182d50',
 '9a27f8ae-069b-446a-b5a6-8015d26482ac',
 '3bb9f843-52c0-4b3e-9a39-cd77db62f961',
 'f1d566f3-bfae-4a70-a852-5bf00e938f35',
 'a3711821-bef9-4d60-b291-d6c0b2a219b7',
 '478d64df-e436-44c3-952f-ad4986831af5',
 '73a36f41-6c76-4993-bb81-515d58d23aa8',
 '1b2bfa30-44b0-4bbc-b172-4a012ecb640e',
 'b341d365-1c3a-4162-a26a-cee8b3ef3750',
 '2e3fd78d-5711-43d2-a3c9-004d0d4baa1e',
 'fac5af2a-208d-48d0-a0d1-53e15f1f04b8',
 'befac585-03a1-4893-b478-b44345531dd5',
 'd1553243-47e9-4ac8-be94-6c0670717fc4',
 'b006d8fa-fbae-43f4-bccf-c6141970d0cc',
 '2513f478-9bee-4a07-a1a9-a784c066fc24',
 '07d4abd6-fd17-4ece-8699-cb8b5468bd5f',
 '73b6dfd9-4095-4868-b27a-fdc5263fc63b',
 '377071b6-a605-4b80-abb5-ced5a7d37925',
 '8ed54aeb-0ec6-42d0-954e-75c096a9c86d',
 '5ac09bd0-9b5b-43a0-a1ae-0682c86fa8e4',
 '3d709f3f-9fb0-4430-babb-35d99c161526',
 '198bde8d-6084-

In [197]:
doc_search = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embeddings, text_key="text")

In [198]:
doc_search

In [199]:
retriever = doc_search.as_retriever(search_kwargs={"k": 3}, search_type="similarity")

In [203]:
retriever_doc = retriever.invoke("What is Acne?")

In [283]:
for i in retriever_doc:
    print(i.page_content)

GALE ENCYCLOPEDIA OF MEDICINE 226
Acne
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26
GALE ENCYCLOPEDIA OF MEDICINE 2 25
Acne
Acne vulgaris affecting a woman’s face. Acne is the general
name given to a skin disorder in which the sebaceous
glands become inflamed.(Photograph by Biophoto Associ-
ates, Photo Researchers, Inc. Reproduced by permission.)
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25
Acidosis see Respiratory acidosis; Renal
tubular acidosis; Metabolic acidosis
Acne
Definition
Acne is a common skin disease characterized by
pimples on the face, chest, and back. It occurs when the
pores of the skin become clogged with oil, dead skin
cells, and bacteria.
Description
Acne vulgaris, the medical term for common acne, is
the most common skin disease. It affects nearly 17 million
people in the United States. While acne can arise at any


In [284]:
from langchain_google_genai import GoogleGenerativeAI
llm = GoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.5, api_key=GOOGLE_API_KEY, max_tokens=1000)

In [285]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [286]:
system_prompt = (
    "You are a medical information specialist that provides thorough, structured answers. "
    "Combine these elements in your response:\n"
    "1. START with a clear definition/overview\n"
    "2. EXPLAIN causes and contributing factors\n"
    "3. DETAIL evidence-based treatments (topical/oral/procedural)\n"
    "4. INCLUDE prevention/management strategies\n"
    "5. NOTE prognosis and cure potential\n"
    "6. MENTION when to consult a professional\n\n"

    "Guidelines:\n"
    "- Integrate retrieved context with medical consensus knowledge\n"
    "- Use layman-friendly terms but maintain scientific accuracy\n"
    "- Present information in logical flow with paragraph breaks\n"
    "- If context is incomplete, supplement with common knowledge (e.g., standard treatments)\n"
    "- Clearly state uncertainties but emphasize actionable advice\n"
    "- Prioritize information utility over strict brevity\n\n"

    "Question: {input}\n"
    "Retrieved Context: {context}\n\n"
)

In [287]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [288]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [289]:
response = rag_chain.invoke({"input": "What is Acne and we can cure it?"})

In [290]:
print(response["answer"])

Okay, let's break down what acne is and how we can manage it.

**1. What is Acne?**

Acne, often called "acne vulgaris," is a common skin condition that happens when hair follicles under the skin get clogged. Sebum (oil produced by sebaceous glands to keep skin moisturized) and dead skin cells plug these pores, creating an environment where bacteria can thrive. This can lead to inflammation and the development of pimples, blackheads, whiteheads, and deeper cysts or nodules.

**2. What Causes Acne?**

Several factors contribute to the development of acne:

*   **Excess Oil Production:** Overactive sebaceous glands produce too much oil, clogging pores.
*   **Clogged Hair Follicles:** Dead skin cells aren't shed properly and accumulate within the hair follicle.
*   **Bacteria:** *Cutibacterium acnes* (formerly *Propionibacterium acnes*) is a bacteria that normally lives on the skin. When trapped in clogged follicles, it multiplies and causes inflammation.
*   **Inflammation:** The body's 